In [3]:
import sys
print(sys.executable)

/Users/charvigulati/Desktop/UWaterloo/Research/FoldSeek/venv/bin/python


In [4]:
import pandas as pd

In [5]:
cols = "query target tm qcov qstart qend evalue".split()
df = pd.read_csv("result_20k.tsv", sep="\t", names=cols)
df["pdb"] = df.target.str[:4]
df["chain"] = df.target.str.split("_").str[-1]

In [6]:
print(df.tm.describe())

count    114732.000000
mean          0.619622
std           0.095522
min           0.290200
25%           0.544700
50%           0.642200
75%           0.697300
max           1.011000
Name: tm, dtype: float64


In [7]:
print((df.tm > 0.7).sum(), "hits above 0.7")

27117 hits above 0.7


In [8]:
print(((df.tm > 0.5) & (df.tm <= 0.7)).sum(), "between 0.5 and 0.7")

77180 between 0.5 and 0.7


In [9]:
knowns = ["1fna", "3tes", "1ten", "4u3h" ] # fibronectin, TenCon, tenascin, FN3con
df[df.pdb.isin(knowns)].sort_values("tm", ascending=False)

,query,target,tm,qcov,qstart,qend,evalue,pdb,chain
0,1FNA,1fna-assembly1_A,1.0110,1.000,1,91,1.0000,1fna,A
387,1FNA,4u3h-assembly1_A,0.9460,0.978,1,89,0.8008,4u3h,A
161,1FNA,3tes-assembly1_A,0.9360,1.000,1,91,0.8541,3tes,A
173,1FNA,3tes-assembly2_B,0.9342,1.000,1,91,0.8518,3tes,B
286,1FNA,3tes-assembly4_D,0.9314,1.000,1,91,0.8477,3tes,D
292,1FNA,3tes-assembly3_C,0.9300,1.000,1,91,0.8463,3tes,C
311,1FNA,1ten-assembly1_A,0.9220,0.978,1,89,0.8342,1ten,A


In [10]:
print((df.tm > 0.9).sum())


430


In [11]:
print((df.tm > 0.85).sum())
print((df.tm > 0.8).sum())

722
1345


In [12]:
hits = df[df.tm > 0.85]
print(hits.pdb.nunique(), "unique PDB entries for a TM score > 0.85")

298 unique PDB entries for a TM score > 0.85


In [13]:
hits = df[df.tm > 0.85].copy()
entries = hits.sort_values("tm", ascending=False).drop_duplicates("pdb")
print(len(entries), "unique entries")

298 unique entries


In [14]:
import requests, time

def fetch(pdb):
    r = requests.get(f"https://data.rcsb.org/rest/v1/core/entry/{pdb}")
    if r.status_code != 200:
        return None
    d = r.json()
    return {
        "pdb": pdb,
        "title": d["struct"]["title"],
        "method": d["rcsb_entry_info"]["experimental_method"],
        "n_entities": d["rcsb_entry_info"]["polymer_entity_count"],
        "n_chains": d["rcsb_entry_info"]["deposited_polymer_entity_instance_count"],
    }

rows = []
for p in entries.pdb:
    rows.append(fetch(p))
    time.sleep(0.1)

meta = pd.DataFrame([r for r in rows if r])
out = entries.merge(meta, on="pdb")
out.to_csv("fn3_homologs_annotated.csv", index=False)

/Users/charvigulati/Desktop/UWaterloo/Research/FoldSeek/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [17]:
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 80)
out[["pdb","tm", "evalue", "title","n_chains", "chain"]]

,pdb,tm,evalue,title,n_chains,chain
0,1fna,1.0110,1.0000,CRYSTAL STRUCTURE OF THE TENTH TYPE III CELL ADHESION MODULE OF HUMAN FIBRON...,1,A
1,6o03,0.9767,0.9105,Monobody (MC17) bound to tyrosine kinase binding domain of E3 ubiquitin liga...,4,D
2,5oc7,0.9758,0.8862,Crystal structure of the pleckstrin-homology domain of Bcr-Abl in complex wi...,4,C
3,8ezg,0.9677,0.9021,Monobody 12D1 bound to KRAS(G12D),2,B
4,3rzw,0.9673,0.8913,Crystal Structure of the Monobody ySMB-9 bound to human SUMO1,4,A
5,8sso,0.9658,0.8714,AurA bound to danusertib and inhibiting monobody Mb2,4,E
6,5nkq,0.9651,0.8625,Crystal structure of a dual topology fluoride ion channel.,8,G
7,8pr7,0.9646,0.8872,Aurora-A in complex with CEP192 and an inhibitory monobody,6,E
8,8gzw,0.9634,0.8812,Klebsiella pneumoniae FtsZ complexed with monobody (P21),6,F
9,8gzv,0.9624,0.8763,Klebsiella pneumoniae FtsZ complexed with monobody (P212121),2,B


In [18]:
import requests, time, json

def chain_desc(pdb, chain):
    url = f"https://data.rcsb.org/rest/v1/core/polymer_entity_instance/{pdb}/{chain}"
    r = requests.get(url)
    if r.status_code != 200:
        return None
    return r.json()

d = chain_desc("8gzw", "F")
print(json.dumps(d, indent=2)[:3000])

{
  "pdbx_vrpt_summary_entity_geometry": [
    {
      "PDB_model_num": 1,
      "angles_RMSZ": 0.58,
      "bonds_RMSZ": 0.33,
      "num_angles_RMSZ": 958,
      "num_bonds_RMSZ": 697
    }
  ],
  "rcsb_polymer_entity_instance_container_identifiers": {
    "asym_id": "F",
    "auth_asym_id": "F",
    "auth_to_entity_poly_seq_mapping": [
      "-1",
      "0",
      "1",
      "2",
      "3",
      "4",
      "5",
      "6",
      "7",
      "8",
      "9",
      "10",
      "11",
      "12",
      "13",
      "14",
      "15",
      "16",
      "17",
      "18",
      "19",
      "20",
      "21",
      "22",
      "23",
      "24",
      "25",
      "26",
      "27",
      "28",
      "29",
      "30",
      "31",
      "32",
      "33",
      "34",
      "35",
      "36",
      "37",
      "38",
      "39",
      "40",
      "41",
      "42",
      "43",
      "44",
      "45",
      "46",
      "47",
      "48",
      "49",
      "50",
      "51",
      "52",
      "53",
      "54

In [21]:
f"https://data.rcsb.org/rest/v1/core/polymer_entity/8gzw/2"

'https://data.rcsb.org/rest/v1/core/polymer_entity/8gzw/2'

In [22]:
import requests

query = {
    "query": {
        "type": "terminal",
        "service": "structure",
        "parameters": {
            "value": {"entry_id": "1FNA", "asym_id": "A"},
            "operator": "relaxed_shape_match",
            "target_search_space": "polymer_entity_instance"
        }
    },
    "return_type": "polymer_instance",
    "request_options": {"return_all_hits": True}
}

response = requests.post("https://search.rcsb.org/rcsbsearch/v2/query", json=query)
data = response.json()

rcsb_ids = []
for hit in data["result_set"]:
    rcsb_ids.append(hit["identifier"])

print(len(rcsb_ids), "hits")

761 hits


In [23]:
knowns = ["1TEN", "1QR4", "3TES", "4U3H"]

for k in knowns:
    found = False
    for rid in rcsb_ids:
        if rid.startswith(k):
            found = True
    print(k, found)

1TEN True
1QR4 True
3TES True
4U3H True


In [24]:
rcsb_pdbs = set()
for rid in rcsb_ids:
    rcsb_pdbs.add(rid[:4].lower())

foldseek_pdbs = set(df[df.tm > 0.85].pdb)

print("both:", len(rcsb_pdbs & foldseek_pdbs))
print("Foldseek only:", len(foldseek_pdbs - rcsb_pdbs))
print("RCSB only:", len(rcsb_pdbs - foldseek_pdbs))

both: 248
Foldseek only: 50
RCSB only: 151


In [25]:
all_foldseek = set(df.pdb)

rcsb_only = rcsb_pdbs - foldseek_pdbs

found_but_low = rcsb_only & all_foldseek
truly_new = rcsb_only - all_foldseek

print("RCSB only, but Foldseek had it below 0.85:", len(found_but_low))
print("RCSB only, Foldseek never found it:", len(truly_new))

RCSB only, but Foldseek had it below 0.85: 139
RCSB only, Foldseek never found it: 12


In [ ]:
print(sorted(truly_new))
print(sorted(foldseek_pdbs - rcsb_pdbs))

['22pl', '8xwy', '9bpu', '9bpv', '9eza', '9iui', '9l14', '9p6s', '9pvo', '9sa8', '9w18', '9y5y']
['1zlg', '2jll', '2obg', '2qbw', '2w1n', '2x10', '2x11', '2xyc', '3ch8', '3csb', '3csg', '3fl7', '4bk4', '4bk5', '4bka', '4bkf', '4jmg', '4jmh', '4lpu', '4lpv', '4lpx', '4lpy', '4m4p', '4m4r', '4pbx', '4y61', '4yfd', '4yfg', '4yh7', '5fmv', '5fn6', '5j6z', '5u8q', '5xwt', '6bt9', '6pyh', '7ceg', '7s7k', '7sl2', '7sti', '7tyk', '7tym', '7xgd', '8avb', '8cls', '8d82', '8dh9', '8tan', '8vjc', '8vse']


In [ ]:
### Let's do the PDB method 

In [ ]:
# out.to_csv("fn3_homologs_annotated.csv", index=False)